# EJERCICIO NLP
## Extracción de N.E.R desde un artículo

Vamos a aplicar lo visto hasta ahora sobre NLP para realizar una extracción de NER sobre un artículo de prensa.

1.- Carga de librerías

In [1]:
import spacy
from spacy import displacy
from collections import Counter
import es_core_news_lg

nlp = spacy.load('es_core_news_lg')

2. Detección de entidades

In [3]:
# pip install html5lib
# pip install bs4
# pip install requests

# html5lib se usa para limpiar el html y bs4 para parsear el html y obtener el texto
# requests se usa para hacer la petición a la página web
# re es para hacer la limpieza de texto

from bs4 import BeautifulSoup
import requests
import re
def url_to_string(url):
    res = requests.get(url)
    html = res.text
    soup = BeautifulSoup(html, 'html5lib')
    for script in soup(["script", "style", 'aside']):
        script.extract()
    return " ".join(re.split(r'[\n\t]+', soup.get_text()))
veinte_min = url_to_string('https://www.20minutos.es/internacional/trump-hablara-martes-putin-desbloquear-iniciativa-paz-eeuu-ucrania-5691587/')
article = nlp(veinte_min)
len(article.ents)

116

Se detectan en este artículo 100 entidades, repartidas como sigue:

In [4]:
labels = [x.label_ for x in article.ents]
Counter(labels)

Counter({'LOC': 39, 'MISC': 30, 'PER': 27, 'ORG': 20})

Las 3 entidades más repetidas son:

In [5]:
items = [x.text for x in article.ents]
Counter(items).most_common(3)

[('Ucrania', 10), ('Putin', 8), ('Trump', 6)]

Análisis de una frase del artículo:

In [6]:
sentences = [x for x in article.sents]
print(sentences[19])

De hecho, el viceministro de Exteriores ruso, Alexander Grushko, ha pedido en una entrevista al periódico Izvestia que la exclusión de Ucrania de la OTAN esté en el acuerdo de paz: "Exigiremos que este acuerdo incluya garantías de seguridad férreas porque solo a través de su formación será posible lograr una paz duradera en Ucrania y fortalecer la seguridad regional en general".


Usamos displaCy para resaltar las entidades:

In [7]:
displacy.render(nlp(str(sentences[19])), jupyter=True, style='ent')

In [8]:
displacy.render(nlp(str(sentences[19])), style='dep', jupyter = True, options = {'distance': 100})

In [9]:
[(x.orth_,x.pos_, x.lemma_) for x in [y 
                                      for y
                                      in nlp(str(sentences[19])) 
                                      if not y.is_stop and y.pos_ != 'PUNCT']]

[('viceministro', 'NOUN', 'viceministro'),
 ('Exteriores', 'PROPN', 'Exteriores'),
 ('ruso', 'ADJ', 'ruso'),
 ('Alexander', 'PROPN', 'Alexander'),
 ('Grushko', 'PROPN', 'Grushko'),
 ('pedido', 'VERB', 'pedir'),
 ('entrevista', 'NOUN', 'entrevista'),
 ('periódico', 'NOUN', 'periódico'),
 ('Izvestia', 'PROPN', 'Izvestia'),
 ('exclusión', 'NOUN', 'exclusión'),
 ('Ucrania', 'PROPN', 'Ucrania'),
 ('OTAN', 'PROPN', 'OTAN'),
 ('esté', 'VERB', 'estar'),
 ('paz', 'NOUN', 'paz'),
 ('Exigiremos', 'VERB', 'exigir'),
 ('incluya', 'VERB', 'incluir'),
 ('garantías', 'NOUN', 'garantía'),
 ('seguridad', 'NOUN', 'seguridad'),
 ('férreas', 'ADJ', 'férreas'),
 ('formación', 'NOUN', 'formación'),
 ('lograr', 'VERB', 'lograr'),
 ('paz', 'NOUN', 'paz'),
 ('duradera', 'ADJ', 'duradero'),
 ('Ucrania', 'PROPN', 'Ucrania'),
 ('fortalecer', 'VERB', 'fortalecer'),
 ('seguridad', 'NOUN', 'seguridad'),
 ('regional', 'ADJ', 'regional'),
 ('general', 'NOUN', 'general')]

In [10]:
dict([(str(x), x.label_) for x in nlp(str(sentences[19])).ents])

{'Exteriores ruso': 'LOC',
 'Alexander Grushko': 'PER',
 'Izvestia': 'ORG',
 'Ucrania': 'LOC',
 'OTAN': 'ORG',
 'Exigiremos': 'LOC'}

In [11]:
print([(x, x.ent_iob_, x.ent_type_) for x in sentences[19]])

[(De, 'O', ''), (hecho, 'O', ''), (,, 'O', ''), (el, 'O', ''), (viceministro, 'O', ''), (de, 'O', ''), (Exteriores, 'B', 'LOC'), (ruso, 'I', 'LOC'), (,, 'O', ''), (Alexander, 'B', 'PER'), (Grushko, 'I', 'PER'), (,, 'O', ''), (ha, 'O', ''), (pedido, 'O', ''), (en, 'O', ''), (una, 'O', ''), (entrevista, 'O', ''), (al, 'O', ''), (periódico, 'O', ''), (Izvestia, 'B', 'ORG'), (que, 'O', ''), (la, 'O', ''), (exclusión, 'O', ''), (de, 'O', ''), (Ucrania, 'B', 'LOC'), (de, 'O', ''), (la, 'O', ''), (OTAN, 'B', 'ORG'), (esté, 'O', ''), (en, 'O', ''), (el, 'O', ''), (acuerdo, 'O', ''), (de, 'O', ''), (paz, 'O', ''), (:, 'O', ''), (", 'O', ''), (Exigiremos, 'B', 'LOC'), (que, 'O', ''), (este, 'O', ''), (acuerdo, 'O', ''), (incluya, 'O', ''), (garantías, 'O', ''), (de, 'O', ''), (seguridad, 'O', ''), (férreas, 'O', ''), (porque, 'O', ''), (solo, 'O', ''), (a, 'O', ''), (través, 'O', ''), (de, 'O', ''), (su, 'O', ''), (formación, 'O', ''), (será, 'O', ''), (posible, 'O', ''), (lograr, 'O', ''), (una

3. Resaltado de todo el artículo.

In [12]:
displacy.render(article, jupyter=True, style='ent')